# S2 — Continued pre-training (multi-seed)

MLM pre-training on the unlabeled corpus (1 epoch, run once), then fine-tune the classifier with seeds 42, 1, 2.

In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))


In [2]:
import json, zipfile
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
from sklearn.metrics import f1_score, classification_report
from huggingface_hub import snapshot_download
import torch
from transformers import (
    AutoTokenizer, AutoModelForMaskedLM, AutoModelForSequenceClassification,
    DataCollatorForLanguageModeling,
    TrainingArguments, Trainer, EarlyStoppingCallback, set_seed,
)
from datasets import Dataset

DATA_DIR   = Path(snapshot_download(repo_id='tamarasuarezrod/longeval-data', repo_type='dataset'))
MODELS_DIR = Path('/tmp/s2')
MLM_DIR    = MODELS_DIR / 'mlm_adapted'
MLM_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'roberta-base'
SEEDS      = [42, 1, 2]
STRATEGY   = 's2-continued-pretraining'
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Device: cuda


## Data

In [3]:
def load_split(path, label_col='label'):
    with open(path) as f:
        records = json.load(f)
    df = pd.DataFrame(records).rename(columns={label_col: 'label'})
    return df[['pp_text', 'label']]

splits = {
    'train':       load_split(DATA_DIR / 'train_eval/train.json',             label_col='distant_label'),
    'eval':        load_split(DATA_DIR / 'train_eval/interim_eval_2016.json', label_col='distant_label'),
    'test_within': load_split(DATA_DIR / 'test/interim_test_2016.json'),
    'test_short':  load_split(DATA_DIR / 'test/interim_test_2018.json'),
    'test_long':   load_split(DATA_DIR / 'test/interim_test_2021.json'),
}
for name, df in splits.items():
    print(f'{name}: {len(df)} rows')

label2id = {l: i for i, l in enumerate(sorted(splits['train']['label'].unique()))}
id2label = {v: k for k, v in label2id.items()}
num_labels = len(label2id)
print('Labels:', label2id)


train: 49608 rows
eval: 1344 rows
test_within: 908 rows
test_short: 908 rows
test_long: 908 rows
Labels: {'negative': 0, 'positive': 1}


## MLM pre-training (run once on unlabeled corpus)

In [4]:

CORPUS_ZIP = DATA_DIR / 'unlabeled_corpus.jl.zip'
texts = []
with zipfile.ZipFile(CORPUS_ZIP) as zf:
    with zf.open(zf.namelist()[0]) as f:
        for line in f:
            rec = json.loads(line)
            t = rec.get('text','').strip()
            if t: texts.append(t)
print(f'Corpus: {len(texts):,} tweets')

mlm_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
corpus_ds = Dataset.from_dict({'text': texts}).map(
    lambda x: mlm_tokenizer(x['text'], truncation=True, max_length=128),
    batched=True, remove_columns=['text'])
data_collator = DataCollatorForLanguageModeling(tokenizer=mlm_tokenizer, mlm=True, mlm_probability=0.15)

mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
mlm_args = TrainingArguments(
    output_dir=str(MLM_DIR), num_train_epochs=1,
    per_device_train_batch_size=32, learning_rate=5e-5, weight_decay=0.01,
    eval_strategy='no', save_strategy='epoch', save_total_limit=1,
    logging_steps=500, fp16=(DEVICE=='cuda'), seed=42, report_to='none')
Trainer(model=mlm_model, args=mlm_args,
        train_dataset=corpus_ds, data_collator=data_collator,
        processing_class=mlm_tokenizer).train()
mlm_model.save_pretrained(str(MLM_DIR))   # save weights + config.json
mlm_tokenizer.save_pretrained(str(MLM_DIR))
print('MLM done. Backbone + tokenizer saved to', MLM_DIR)


Corpus: 1,079,995 tweets


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
500,6.454043
1000,6.278521
1500,6.127505
2000,6.091962
2500,6.001878
3000,5.942793
3500,5.937055
4000,5.946005
4500,5.876797
5000,5.849282


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MLM done. Backbone + tokenizer saved to /tmp/s2/mlm_adapted


In [5]:
# Push MLM backbone to HF immediately — if the classifier loop fails later,
# reload with: mlm_model = AutoModelForMaskedLM.from_pretrained('tamarasuarezrod/longeval-s2-mlm-backbone')
MLM_REPO = 'tamarasuarezrod/longeval-s2-mlm-backbone'
mlm_model.push_to_hub(MLM_REPO)
mlm_tokenizer.push_to_hub(MLM_REPO)
print('MLM backbone saved to HF:', MLM_REPO)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

MLM backbone saved to HF: tamarasuarezrod/longeval-s2-mlm-backbone


## Classifier fine-tuning loop (seeds 42, 1, 2)

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)  # tokenizer unchanged by MLM

def make_dataset(df):
    d = Dataset.from_dict({
        'text':  df['pp_text'].tolist(),
        'label': df['label'].map(label2id).tolist(),
    })
    return d.map(
        lambda x: tokenizer(x['text'], truncation=True, padding='max_length', max_length=128),
        batched=True, remove_columns=['text'],
    )

eval_ds  = make_dataset(splits['eval'])
train_ds = make_dataset(splits['train'])
test_ds  = {n: make_dataset(df) for n, df in splits.items() if n not in ('train','eval')}

all_results  = {}
saved_models = {}

for SEED in SEEDS:
    print(f'\n{"="*50}')
    print(f'SEED {SEED}')
    print('='*50)
    set_seed(SEED)
    clf_dir = MODELS_DIR / f'clf_seed{SEED}'
    clf_dir.mkdir(parents=True, exist_ok=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MLM_REPO, num_labels=num_labels, id2label=id2label, label2id=label2id)

    args = TrainingArguments(
        output_dir=str(clf_dir), num_train_epochs=25,
        per_device_train_batch_size=32, per_device_eval_batch_size=64,
        learning_rate=2e-05, warmup_ratio=0.1, weight_decay=0.01,
        eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
        logging_steps=50, fp16=(DEVICE=='cuda'), seed=SEED, report_to='none')

    trainer = Trainer(model=model, args=args,
                      train_dataset=train_ds, eval_dataset=eval_ds,
                      processing_class=tokenizer,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    trainer.train()

    seed_results = {}
    for name, ds in test_ds.items():
        preds_out = trainer.predict(ds)
        y_pred = np.argmax(preds_out.predictions, axis=1)
        f1 = f1_score(np.array(ds['label']), y_pred, average='macro')
        seed_results[name] = f1
        print(f'  {name}: F1={f1:.4f}')

    all_results[SEED]  = seed_results
    saved_models[SEED] = trainer.model


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  test_within: F1=0.7001


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


  test_short: F1=0.6724


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


  test_long: F1=0.6709


## Summary

In [6]:
# ── Summary ────────────────────────────────────────────────
import numpy as np

splits = ['test_within', 'test_short', 'test_long']
print(f"{'seed':>8}  {'within':>7}  {'short':>7}  {'long':>7}  {'RPD_short':>10}  {'RPD_long':>9}")
for seed, r in all_results.items():
    rpd_s = (r['test_short']  - r['test_within']) / r['test_within']
    rpd_l = (r['test_long']   - r['test_within']) / r['test_within']
    print(f"{seed:>8}  {r['test_within']:>7.4f}  {r['test_short']:>7.4f}  {r['test_long']:>7.4f}  {rpd_s:>+10.4f}  {rpd_l:>+9.4f}")

print()
for sp in splits:
    vals = [all_results[s][sp] for s in all_results]
    print(f"{sp}: mean={np.mean(vals):.4f}  std={np.std(vals,ddof=1):.4f}  [{min(vals):.4f}–{max(vals):.4f}]")


    seed   within    short     long   RPD_short   RPD_long
      42   0.7203   0.6795   0.6915     -0.0566    -0.0399
       1   0.7421   0.6770   0.7103     -0.0878    -0.0429
       2   0.7001   0.6724   0.6709     -0.0395    -0.0417

test_within: mean=0.7208  std=0.0210  [0.7001–0.7421]
test_short: mean=0.6763  std=0.0036  [0.6724–0.6795]
test_long: mean=0.6909  std=0.0197  [0.6709–0.7103]


## Save to HuggingFace

In [7]:
for seed, model in saved_models.items():
    repo = f'tamarasuarezrod/longeval-{STRATEGY}-seed{seed}'
    model.push_to_hub(repo)
    tokenizer.push_to_hub(repo)
    print('Saved:', repo)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Saved: tamarasuarezrod/longeval-s2-continued-pretraining-seed42


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Saved: tamarasuarezrod/longeval-s2-continued-pretraining-seed2
